# Example 2: With Cell Metadata - Stroke Dataset

This notebook demonstrates processing a modern GEO dataset where each sample has paired count and cell metadata files.

**Dataset**: GSE225948 (Stroke brain and blood samples)  
**Format**: Paired *_counts.csv.gz and *_metadata.csv.gz files per sample  
**Samples**: 3 samples (2 brain, 1 blood)

In [ ]:
import sys
import os
sys.path.append('../../')  # Add repo to path

from anndata_compiler import GEOAnndataCompiler
import pandas as pd

## 1. Examine the Data Structure

In [ ]:
# Look at the files
data_dir = '../data/with_cell_metadata_stroke'
print("Files in data directory:")
files = sorted(os.listdir(data_dir))
for f in files:
    print(f"  {f}")

print(f"\nFile pairs:")
counts_files = [f for f in files if '_counts.csv.gz' in f]
metadata_files = [f for f in files if '_metadata.csv.gz' in f]
print(f"  Count files: {len(counts_files)}")
print(f"  Metadata files: {len(metadata_files)}")

In [ ]:
# Look at sample-level metadata
metadata = pd.read_csv(f'{data_dir}/metadata.csv')
print(f"Sample-level metadata ({metadata.shape[0]} samples):")
print(metadata)

## 2. Configure the Compiler

The compiler will auto-detect the paired file format:

In [ ]:
config = {
    'raw_data_dir': data_dir,
    'metadata_file': f'{data_dir}/metadata.csv',
    'output_file': './stroke_compiled_example.h5ad',
    'sample_id_column': 'Sample_ID',
    
    # Processing parameters
    'max_cells_per_sample': 300,  # Small for demo
    'target_sum': 1e4,
    'n_top_genes': 2000,
    'delimiter': ',',  # CSV files
    'optimize_params': True,
    
    # Auto-detection will find the paired files
    'data_format': 'auto',  # Will detect 'with_cell_metadata'
    
    # Selective metadata inclusion
    'metadata_columns': ['Tissue', 'Condition', 'Sex']
}

print("Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 3. Run the Compilation Pipeline

In [ ]:
# Initialize compiler
compiler = GEOAnndataCompiler(config)

# Run full pipeline
adata = compiler.run_full_pipeline(
    plot_colors=['leiden', 'Tissue', 'sample_id']
)

print(f"\nFinal dataset: {adata.n_obs} cells × {adata.n_vars} genes")
print(f"Samples: {adata.obs['sample_id'].unique()}")
print(f"Tissues: {adata.obs['Tissue'].unique()}")

## 4. Explore Cell Metadata Integration

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# Basic info
print("AnnData structure:")
print(adata)
print(f"\nObservation columns: {list(adata.obs.columns)}")

# Check what cell metadata was preserved from GEO
cell_meta_cols = [col for col in adata.obs.columns if col not in ['sample_id', 'Tissue', 'Condition', 'Sex']]
print(f"\nCell metadata from GEO files: {cell_meta_cols}")

In [ ]:
# Sample and tissue composition
composition = adata.obs.groupby(['sample_id', 'Tissue']).size().unstack(fill_value=0)
print("Cells per sample and tissue:")
print(composition)

# If cell types were annotated in the GEO metadata, show them
if 'cell_type' in adata.obs.columns:
    print(f"\nCell types found: {adata.obs['cell_type'].unique()}")
    cell_type_counts = adata.obs['cell_type'].value_counts()
    print(cell_type_counts)

In [ ]:
# UMAP visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, frameon=False)
axes[0].set_title('Leiden Clusters')

sc.pl.umap(adata, color='Tissue', ax=axes[1], show=False, frameon=False)
axes[1].set_title('Tissue Type')

sc.pl.umap(adata, color='sample_id', ax=axes[2], show=False, frameon=False)
axes[2].set_title('Sample ID')

plt.tight_layout()
plt.show()

In [ ]:
# If cell types are available, plot them too
if 'cell_type' in adata.obs.columns:
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    sc.pl.umap(adata, color='cell_type', ax=ax, show=False, frameon=False)
    ax.set_title('Cell Types (from GEO metadata)')
    plt.show()

## 5. Compare Sample-level vs Cell-level Metadata

In [ ]:
# Show how both levels of metadata are preserved
print("Sample-level metadata (added to all cells in each sample):")
sample_meta = adata.obs[['sample_id', 'Tissue', 'Condition', 'Sex']].drop_duplicates()
print(sample_meta)

print("\nExample cell-level metadata (varies per cell):")
if len(cell_meta_cols) > 0:
    example_cols = ['sample_id'] + cell_meta_cols[:3]  # Show first 3 cell metadata columns
    print(adata.obs[example_cols].head(10))
else:
    print("No cell-level metadata columns found in this example.")

## Summary

This example demonstrated:
- Auto-detection of paired count/metadata files
- Integration of both sample-level and cell-level metadata
- Processing modern GEO datasets with rich annotations
- Selective inclusion of sample metadata columns
- Preservation of cell type annotations from GEO

The final AnnData object contains both your custom sample metadata and the original cell-level annotations from the GEO submission!